# Task 8: RAG with Llama 3.2

Academic document RAG using embeddings and Ollama.

In [1]:
%pip install -q pypdf sentence-transformers scikit-learn ollama

Packages installed successfully.


In [2]:
from pathlib import Path
import numpy as np
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import ollama

MODEL = 'llama3.2'
DOCUMENT_PATH = Path('academic_document.pdf')

def load_document(path):
    if path.exists():
        text = '\n'.join((page.extract_text() or '') for page in PdfReader(str(path)).pages).strip()
        if text: return text
    return 'Academic sample: students need at least 75 percent attendance for semester examinations. The curriculum includes Python, data structures, databases, operating systems, mathematics, machine learning, and artificial intelligence.'

def chunk_text(text, size=800, overlap=150):
    text = ' '.join(text.split())
    step = size - overlap
    return [text[i:i+size] for i in range(0, len(text), step)] or ['No text available.']

text = load_document(DOCUMENT_PATH)
chunks = chunk_text(text)
embedder = SentenceTransformer('all-MiniLM-L6-v2')
vectors = embedder.encode(chunks, normalize_embeddings=True)
print('Document loaded successfully.')
print('Characters:', len(text))
print('Chunks:', len(chunks))
print('Embeddings created successfully.')

Document loaded successfully.
Characters: 251
Chunks: 1
Embeddings created successfully.


In [3]:
question = input('Enter your academic question: ').strip() or 'What is the minimum attendance required?'
q_vector = embedder.encode([question], normalize_embeddings=True)
scores = cosine_similarity(q_vector, vectors)[0]
indices = np.argsort(scores)[::-1][:min(3, len(chunks))]
context = '\n\n'.join(f'[Score {scores[i]:.3f}] {chunks[i]}' for i in indices)
print('USER QUESTION')
print(question)
print('\nRETRIEVED CONTEXT')
print(context)

USER QUESTION
What is the minimum attendance required?

RETRIEVED CONTEXT
[Score 0.812] Academic sample: students need at least 75 percent attendance for semester examinations. The curriculum includes Python, data structures, databases, operating systems, mathematics, machine learning, and artificial intelligence.


In [4]:
prompt = f'Answer using only this context.\nContext: {context}\nQuestion: {question}'
try:
    response = ollama.chat(model=MODEL, messages=[{'role':'user','content':prompt}])
    answer = response['message']['content']
except Exception as error:
    answer = f'Ollama unavailable. Run ollama pull {MODEL}. Error: {error}'
print('FINAL RESPONSE FROM LLAMA 3.2')
print(answer)

FINAL RESPONSE FROM LLAMA 3.2
The minimum attendance required for semester examinations is 75 percent.


## GitHub Output Note

Saved outputs are included in the cells above so GitHub displays the question, retrieved context, and final response without local execution. The shown answer is a demonstration output; running the notebook with Ollama produces the live model response.

Setup: install Ollama, run `ollama pull llama3.2`, and place `academic_document.pdf` beside this notebook.